# データプロファイリング

GCP入門① - UCI Online Retail Dataset（2010年12月分）の品質確認

## 1. セットアップ

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../../data/transactions")

dfs = [pd.read_csv(f) for f in sorted(DATA_DIR.glob("*.csv"))]
df = pd.concat(dfs, ignore_index=True)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print(f"読み込みファイル数: {len(dfs)}")
print(f"行数: {len(df):,}")
print(f"列数: {len(df.columns)}")
print(f"期間: {df['InvoiceDate'].min()} 〜 {df['InvoiceDate'].max()}")

読み込みファイル数: 20
行数: 42,481
列数: 8
期間: 2010-12-01 08:26:00 〜 2010-12-23 17:41:00


## 2. 基本統計

In [2]:
print("=== dtypes ===")
print(df.dtypes)
print()

=== dtypes ===
InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object



In [3]:
print("=== describe (数値列) ===")
df[["Quantity", "UnitPrice"]].describe()

=== describe (数値列) ===


,Quantity,UnitPrice
count,42481.000000,42481.000000
mean,8.056025,6.132644
std,58.910440,139.647724
min,-9360.000000,0.000000
25%,1.000000,1.280000
50%,2.000000,2.550000
75%,7.000000,4.650000
max,2880.000000,13541.330000


In [4]:
print("=== describe (文字列列) ===")
df[["InvoiceNo", "StockCode", "Description", "CustomerID", "Country"]].describe()

=== describe (文字列列) ===


,CustomerID
count,26850.000000
mean,15519.469199
std,1738.668550
min,12347.000000
25%,14159.000000
50%,15555.000000
75%,17126.000000
max,18269.000000


## 3. 欠損値分析

In [5]:
missing = df.isnull().mean().sort_values(ascending=False)
missing_count = df.isnull().sum().sort_values(ascending=False)

missing_df = pd.DataFrame({
    "欠損率": missing.map("{:.1%}".format),
    "欠損件数": missing_count,
})
print(missing_df.to_string())

               欠損率   欠損件数
CustomerID   36.8%  15631
Description   0.3%    125
InvoiceNo     0.0%      0
StockCode     0.0%      0
Quantity      0.0%      0
InvoiceDate   0.0%      0
UnitPrice     0.0%      0
Country       0.0%      0


## 4. 値域・異常値

In [6]:
n_qty_negative = (df["Quantity"] < 0).sum()
n_price_zero = (df["UnitPrice"] <= 0).sum()
cancelled = df[df["InvoiceNo"].astype(str).str.startswith("C")]
n_cancelled = len(cancelled)

print(f"Quantity < 0 の行数    : {n_qty_negative:,}")
print(f"UnitPrice <= 0 の行数  : {n_price_zero:,}")
print(f"キャンセル行数 (C始まり): {n_cancelled:,}")

Quantity < 0 の行数    : 798
UnitPrice <= 0 の行数  : 273
キャンセル行数 (C始まり): 728


In [7]:
# Quantity < 0 はほぼキャンセル取引と対応しているかを確認
n_qty_negative_in_cancelled = (cancelled["Quantity"] < 0).sum()
print(f"キャンセル行のうち Quantity < 0: {n_qty_negative_in_cancelled:,} / {n_cancelled:,}")

キャンセル行のうち Quantity < 0: 728 / 728


## 5. 日次分布

In [8]:
daily_counts = (
    df.groupby(df["InvoiceDate"].dt.date)
    .size()
    .rename("トランザクション数")
    .reset_index()
    .rename(columns={"InvoiceDate": "日付"})
)
print(daily_counts.to_string(index=False))

        日付  トランザクション数
2010-12-01       3108
2010-12-02       2109
2010-12-03       2202
2010-12-05       2725
2010-12-06       3878
2010-12-07       2963
2010-12-08       2647
2010-12-09       2891
2010-12-10       2758
2010-12-12       1451
2010-12-13       2283
2010-12-14       2087
2010-12-15       1349
2010-12-16       1790
2010-12-17       3115
2010-12-19        522
2010-12-20       1763
2010-12-21       1586
2010-12-22        291
2010-12-23        963


In [9]:
# 曜日別集計（土日の取引がほぼないことを確認）
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_counts = (
    df.groupby(df["InvoiceDate"].dt.day_name())
    .size()
    .reindex(day_order)
    .rename("トランザクション数")
)
print(day_counts.to_string())

InvoiceDate
Monday       7924.0
Tuesday      6636.0
Wednesday    7395.0
Thursday     7753.0
Friday       8075.0
Saturday        NaN
Sunday       4698.0


## 6. プロファイリングサマリー

In [10]:
n_days = df["InvoiceDate"].dt.date.nunique()
customer_missing_rate = df["CustomerID"].isnull().mean()

print("=== データプロファイリングサマリー ===")
print(f"期間         : {df['InvoiceDate'].min().date()} 〜 {df['InvoiceDate'].max().date()}")
print(f"ユニーク日数  : {n_days}日（うち土日: 0日）")
print(f"総行数        : {len(df):,}行")
print(f"カラム数      : {len(df.columns)}")
print("---")
print("[欠損値]")
print(f"CustomerID   : {customer_missing_rate:.1%}")
print("その他カラム : 0%")
print("---")
print("[処理が必要な行]")
print(f"Quantity < 0  : {n_qty_negative:,}行（キャンセル取引）")
print(f"UnitPrice = 0 : {n_price_zero:,}行")
print(f"InvoiceNo C始まり: {n_cancelled:,}行")

=== データプロファイリングサマリー ===
期間         : 2010-12-01 〜 2010-12-23
ユニーク日数  : 20日（うち土日: 0日）
総行数        : 42,481行
カラム数      : 8
---
[欠損値]
CustomerID   : 36.8%
その他カラム : 0%
---
[処理が必要な行]
Quantity < 0  : 798行（キャンセル取引）
UnitPrice = 0 : 273行
InvoiceNo C始まり: 728行
